# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
alchemy_homo_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy_homo'
alchemy_lumo_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy_lumo'
aqsol_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol'
orderly_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction'
presto_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction'
orderly_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-retrosynthesis'
presto_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-retrosynthesis'
lpm_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_lpm-24'

In [3]:
# load data
alchemy_homo_data = datasets.load_from_disk(alchemy_homo_path)
alchemy_lumo_data = datasets.load_from_disk(alchemy_lumo_path)
aqsol_data = datasets.load_from_disk(aqsol_path)
orderly_forward_data = datasets.load_from_disk(orderly_forward_path)
presto_forward_data = datasets.load_from_disk(presto_forward_path)
orderly_retro_data = datasets.load_from_disk(orderly_retro_path)
presto_retro_data = datasets.load_from_disk(presto_retro_path)
lpm_data = datasets.load_from_disk(lpm_path)


In [4]:
ood_data = datasets.concatenate_datasets(
    [
        alchemy_homo_data,
        aqsol_data,
        orderly_forward_data,
        presto_forward_data,
    ]
)

In [5]:
ood_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 3929
})

In [6]:
ood_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood-v2'
)

Saving the dataset (1/1 shards): 100%|██████████| 3929/3929 [00:00<00:00, 10221.08 examples/s]


In [7]:
augmented_ood_data = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood-v2_augmented')

In [8]:
test_data_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_augmented_0211'
test_data = datasets.load_from_disk(test_data_path)

In [9]:
test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string', '0-th_rejected_x', '0-th_rejected_edge_index', '0-th_rejected_edge_attr', '0-th_additional_rejected_x', '0-th_additional_rejected_edge_index', '0-th_additional_rejected_edge_attr', '1-th_rejected_x', '1-th_rejected_edge_index', '1-th_rejected_edge_attr', '1-th_additional_rejected_x', '1-th_additional_rejected_edge_index', '1-th_additional_rejected_edge_attr', '2-th_rejected_x', '2-th_rejected_edge_index', '2-th_rejected_edge_attr', '2-th_additional_rejected_x', '2-th_additional_rejected_edge_index', '2-th_additional_rejected_edge_attr', '3-th_rejected_x', '3-th_rejected_edge_index', '3-th_rejected_edge_attr', '3-th_additional_rejected_x', '3-th_additional_rejected_edge_index', '3-th_additional_rejected_edge_attr', '4-th_rejected_x', '4-th_rejected_edge_index', '4-th_rejected_edge_attr', '4-th_additional_rejec

In [10]:
ood_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy_homo',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction',
]
test_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-text2mol_0219',
]

train_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-text2mol_0219',
]


In [11]:
test_data = datasets.concatenate_datasets(
    [datasets.load_from_disk(path) for path in test_paths + ood_paths]
)

In [12]:
test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 20205
})

In [17]:
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ablation_0224'
)
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_validation_ablation_0224'
)

Saving the dataset (0/1 shards):   0%|          | 0/20205 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 20205/20205 [00:01<00:00, 11982.98 examples/s]


In [14]:
train_data = datasets.concatenate_datasets(
    [datasets.load_from_disk(path) for path in train_paths]
)

In [15]:
train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 361115
})

In [18]:
train_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_ablation_0224'
)

Saving the dataset (0/7 shards):   0%|          | 0/361115 [00:00<?, ? examples/s]

Saving the dataset (7/7 shards): 100%|██████████| 361115/361115 [00:28<00:00, 12500.32 examples/s]
